# 01 — Exploratory Data Analysis: NYC Restaurant Landscape

Understand coverage, quality, and distributions of every data source before feature engineering.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

In [ ]:
from src.config import get_settings
from src.data.registry import DATASET_REGISTRY

settings = get_settings()
print("Project:", settings.project_name)
print("Registered datasets:", len(DATASET_REGISTRY))
for name, spec in DATASET_REGISTRY.items():
    print(f"  {name:20s}  owner={spec.owner:12s}  status={spec.status}")

## Dataset Coverage Audit


In [ ]:
from src.data.audit import build_default_audit_rows

audit_df = pd.DataFrame([r.model_dump() for r in build_default_audit_rows()])
print(
    audit_df[["name", "owner", "spatial_unit", "time_grain", "status"]].to_string(
        index=False
    )
)

In [ ]:
status_counts = audit_df["status"].value_counts()
fig, ax = plt.subplots(figsize=(7, 3))
status_counts.plot.barh(ax=ax, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_xlabel("Count")
ax.set_title("Dataset Status Distribution")
plt.tight_layout()
plt.show()

## License Activity


In [ ]:
from src.data.etl_licenses import run_etl as load_licenses

lic = load_licenses()
print("Shape:", lic.shape)
display(lic.head())
print("\nStatus counts:")
print(lic["license_status"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
lic["license_status"].value_counts().plot.bar(ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("License Status Distribution")
ax.set_xlabel("Status")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Inspection Grades


In [ ]:
from src.data.etl_inspections import run_etl as load_inspections

insp = load_inspections()
print("Shape:", insp.shape)
display(insp.head())

In [ ]:
grade_counts = insp["grade"].value_counts().head(6)
fig, ax = plt.subplots(figsize=(6, 3))
grade_counts.plot.bar(ax=ax, color="#55A868", edgecolor="white")
ax.set_title("Inspection Grade Distribution")
ax.set_xlabel("Grade")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## ACS — Demographic Context


In [ ]:
from src.data.etl_acs import run_etl as load_acs

acs = load_acs()
print("Shape:", acs.shape)
print(acs.describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, col, title in zip(
    axes,
    ["median_income", "population", "rent_burden"],
    ["Median Income", "Population", "Rent Burden"],
):
    ax.hist(acs[col].dropna(), bins=20, color="#4C72B0", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel(col)
plt.suptitle("ACS Distribution by NTA", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## Key Findings

- All 12 data sources are registered and produce synthetic data when live APIs are unavailable.
- License activity covers active, issued, expired, and revoked statuses.
- Inspection grades skew toward A/B — lower grades are risk signals for the survival model.
- ACS median income ranges broadly across NTAs — rent_burden is the tightest [0.25–0.55].
- Next step: compute permit velocity and license net change for the feature matrix.
